# Topo-Brain: Data Preprocessing with 3T→7T Registration

**Phase 1: Fix Data Alignment**

This notebook preprocesses raw NIfTI MRI data with proper 3T→7T registration.

**Instructions:**
1. Upload your raw `Nifti/` folder to Google Drive (`MyDrive/Nifti/`)
2. Click **Runtime → Change runtime type** → Select **T4 GPU** → Save
3. Run each cell in order
4. Preprocessed data will be saved to Google Drive

In [ ]:
# 1️⃣ Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2️⃣ Verify Raw Data Location
import os

DATA_ROOT = '/content/drive/MyDrive/Nifti'

if os.path.exists(DATA_ROOT):
    subjects = sorted([d for d in os.listdir(DATA_ROOT) if d.startswith('sub-')])
    print(f"✓ Found {len(subjects)} subjects: {subjects}")
    
    # Check first subject structure
    sub1 = os.path.join(DATA_ROOT, subjects[0])
    sessions = os.listdir(sub1)
    print(f"✓ Sessions: {sessions}")
    
    if 'ses-1' in sessions and 'ses-2' in sessions:
        print("✓ Both 3T (ses-1) and 7T (ses-2) sessions found!")
    else:
        print("⚠️ Warning: Expected ses-1 and ses-2 folders")
else:
    print(f"❌ ERROR: Data not found at {DATA_ROOT}")
    print("Please upload your Nifti folder to Google Drive root!")

In [ ]:
# 3️⃣ Clone Repository
import os
os.chdir('/content')
!rm -rf /content/Topo-Brain
!git clone -b colab-ready https://github.com/prabeshx12/Topo-Brain.git /content/Topo-Brain

In [ ]:
# 4️⃣ Install Dependencies
%pip install -q SimpleITK monai nibabel tqdm

In [ ]:
# 5️⃣ Setup Paths and Imports
import sys
sys.path.insert(0, '/content/Topo-Brain')

import numpy as np
import nibabel as nib
from pathlib import Path
import logging
import json
from tqdm import tqdm

from src.preprocessing import ImageRegistration, IntensityNormalization, SkullStripping

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

print("✓ Imports successful!")

In [ ]:
# 6️⃣ Configuration

# Input/Output paths
DATA_ROOT = Path('/content/drive/MyDrive/Nifti')
OUTPUT_ROOT = Path('/content/preprocessed_registered')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# Preprocessing settings
MODALITY = 'T1w'  # or 'T2w'
USE_REGISTRATION = True
REGISTRATION_TYPE = 'rigid'  # 'rigid' or 'affine'
NORMALIZATION = 'zscore'

print(f"Input: {DATA_ROOT}")
print(f"Output: {OUTPUT_ROOT}")
print(f"Modality: {MODALITY}")
print(f"Registration: {REGISTRATION_TYPE}")

In [ ]:
# 7️⃣ Discover Data Pairs

def discover_pairs(data_root, modality='T1w'):
    """Find matching 3T-7T pairs for each subject."""
    pairs = []
    
    for sub_dir in sorted(data_root.glob('sub-*')):
        subject_id = sub_dir.name
        
        # Find 3T (ses-1) and 7T (ses-2) files
        ses1_pattern = f'{subject_id}_ses-1_{modality}*.nii.gz'
        ses2_pattern = f'{subject_id}_ses-2_{modality}*.nii.gz'
        
        ses1_files = list((sub_dir / 'ses-1' / 'anat').glob(ses1_pattern))
        ses2_files = list((sub_dir / 'ses-2' / 'anat').glob(ses2_pattern))
        
        if ses1_files and ses2_files:
            pairs.append({
                'subject': subject_id,
                'path_3t': ses1_files[0],
                'path_7t': ses2_files[0],
            })
            logger.info(f"Found pair for {subject_id}")
        else:
            logger.warning(f"Missing data for {subject_id}: 3T={len(ses1_files)}, 7T={len(ses2_files)}")
    
    return pairs

pairs = discover_pairs(DATA_ROOT, MODALITY)
print(f"\n✓ Found {len(pairs)} complete 3T-7T pairs")

In [ ]:
# 8️⃣ Preprocessing Function

def preprocess_pair(pair, output_root, registration_type='rigid'):
    """
    Preprocess a 3T-7T pair:
    1. Register 3T to 7T space
    2. Apply skull stripping (if masks available)
    3. Normalize intensities
    4. Save both volumes
    """
    subject = pair['subject']
    path_3t = pair['path_3t']
    path_7t = pair['path_7t']
    
    logger.info(f"Processing {subject}...")
    
    # Initialize components
    registrar = ImageRegistration(transform_type=registration_type, num_iterations=200)
    normalizer = IntensityNormalization(method='zscore')
    
    # Step 1: Register 3T to 7T
    registered_3t, fixed_7t, affine = registrar.register(path_3t, path_7t)
    
    # Step 2: Normalize intensities (on brain region only)
    registered_3t = normalizer(registered_3t)
    fixed_7t = normalizer(fixed_7t)
    
    # Step 3: Save outputs maintaining BIDS-like structure
    for session, data, original_path in [
        ('ses-1', registered_3t, path_3t),
        ('ses-2', fixed_7t, path_7t)
    ]:
        out_dir = output_root / subject / session / 'anat'
        out_dir.mkdir(parents=True, exist_ok=True)
        
        # Output filename
        out_name = original_path.name.replace('_defaced', '_preprocessed')
        out_path = out_dir / out_name
        
        # Save NIfTI
        nib.save(nib.Nifti1Image(data, affine), str(out_path))
        
        # Save metadata
        meta = {
            'original_path': str(original_path),
            'shape': list(data.shape),
            'registration': registration_type if session == 'ses-1' else 'reference',
            'normalization': 'zscore'
        }
        meta_path = out_dir / f"{out_path.stem.replace('.nii', '')}_metadata.json"
        with open(meta_path, 'w') as f:
            json.dump(meta, f, indent=2)
        
        logger.info(f"  Saved: {out_path.name} (shape: {data.shape})")
    
    return True

print("✓ Preprocessing function defined")

In [ ]:
# 9️⃣ Run Preprocessing Pipeline
# ⚠️ This takes 30-60 minutes for 10 subjects

print(f"Processing {len(pairs)} subjects...")
print("="*50)

successful = 0
failed = []

for pair in tqdm(pairs, desc="Preprocessing"):
    try:
        preprocess_pair(pair, OUTPUT_ROOT, REGISTRATION_TYPE)
        successful += 1
    except Exception as e:
        logger.error(f"Failed to process {pair['subject']}: {e}")
        failed.append(pair['subject'])

print("="*50)
print(f"✓ Successfully processed: {successful}/{len(pairs)}")
if failed:
    print(f"✗ Failed: {failed}")

In [ ]:
# 🔟 Verify Registration Quality (Visual Check)
import matplotlib.pyplot as plt

def visualize_registration(subject_id, output_root):
    """Show 3T and 7T side-by-side to verify alignment."""
    path_3t = list((output_root / subject_id / 'ses-1' / 'anat').glob('*_preprocessed.nii.gz'))[0]
    path_7t = list((output_root / subject_id / 'ses-2' / 'anat').glob('*_preprocessed.nii.gz'))[0]
    
    img_3t = nib.load(str(path_3t)).get_fdata()
    img_7t = nib.load(str(path_7t)).get_fdata()
    
    # Get middle slices
    mid_x = img_3t.shape[0] // 2
    mid_y = img_3t.shape[1] // 2
    mid_z = img_3t.shape[2] // 2
    
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    fig.suptitle(f'{subject_id}: Registration Verification', fontsize=14)
    
    # 3T row
    axes[0, 0].imshow(img_3t[mid_x, :, :].T, cmap='gray', origin='lower')
    axes[0, 0].set_title('3T Sagittal')
    axes[0, 1].imshow(img_3t[:, mid_y, :].T, cmap='gray', origin='lower')
    axes[0, 1].set_title('3T Coronal')
    axes[0, 2].imshow(img_3t[:, :, mid_z].T, cmap='gray', origin='lower')
    axes[0, 2].set_title('3T Axial')
    
    # 7T row
    axes[1, 0].imshow(img_7t[mid_x, :, :].T, cmap='gray', origin='lower')
    axes[1, 0].set_title('7T Sagittal')
    axes[1, 1].imshow(img_7t[:, mid_y, :].T, cmap='gray', origin='lower')
    axes[1, 1].set_title('7T Coronal')
    axes[1, 2].imshow(img_7t[:, :, mid_z].T, cmap='gray', origin='lower')
    axes[1, 2].set_title('7T Axial')
    
    for ax in axes.flat:
        ax.axis('off')
    
    plt.tight_layout()
    plt.show()
    
    print(f"3T shape: {img_3t.shape}")
    print(f"7T shape: {img_7t.shape}")
    print(f"Shapes match: {img_3t.shape == img_7t.shape}")

# Visualize first subject
if successful > 0:
    visualize_registration(pairs[0]['subject'], OUTPUT_ROOT)

In [ ]:
# 1️⃣1️⃣ Copy to Google Drive
import shutil

DRIVE_OUTPUT = Path('/content/drive/MyDrive/preprocessed_registered')

print(f"Copying preprocessed data to Google Drive...")
print(f"From: {OUTPUT_ROOT}")
print(f"To: {DRIVE_OUTPUT}")

# Remove old if exists
if DRIVE_OUTPUT.exists():
    shutil.rmtree(DRIVE_OUTPUT)

shutil.copytree(OUTPUT_ROOT, DRIVE_OUTPUT)

print(f"\n✓ Copied to Google Drive!")
print(f"\nData location: {DRIVE_OUTPUT}")

In [ ]:
# 1️⃣2️⃣ Create Tar Archive for Training
import subprocess

TAR_PATH = '/content/drive/MyDrive/preprocessed_registered.tar'

print("Creating tar archive...")
os.chdir('/content')
subprocess.run(['tar', '-cvf', TAR_PATH, 'preprocessed_registered'], check=True)

# Get file size
size_mb = os.path.getsize(TAR_PATH) / (1024 * 1024)
print(f"\n✓ Created: {TAR_PATH}")
print(f"Size: {size_mb:.1f} MB")

## ✅ Preprocessing Complete!

Your registered and preprocessed data is saved to:
- **Folder**: `MyDrive/preprocessed_registered/`
- **Archive**: `MyDrive/preprocessed_registered.tar`

### Next Steps (Phase 2):
1. Update `train_colab.ipynb` to use the new data
2. Modify training script with discriminator fixes
3. Run GAN training with registered data